# 比较情绪分析结果（脚本版）

这个 notebook 使用 `compare_results.py` 里的封装函数来复现 `compare_results.ipynb` 的呈现逻辑。

分析内容包括：

- 各结果文件的样本量与字段检查
- 不同结果之间 `id` 交集规模统计
- 各 pair 在交集样本上的 `sentiment_score` 与 `confidence` 对比
- 差值分布、散点图和相关性热图
- 各 pair 差异最大的样本明细

当前默认配置为比较：`4_v2_gemini`、`4_structured_input_gemini`、`4_golden_structured_input_gemini`、`4_golden_structured_input_gpt`。

后续如果你新增了结果目录，只需要修改下一格里的 `RESULT_DATASETS` 和 `PAIRS_TO_COMPARE`。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

import compare_results as cr

plt.style.use("seaborn-v0_8")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

BASE_DIR = Path(r"D:\AQUMON\data\results\4")

RESULT_DATASETS = [
    "4_v2_gemini",
    "4_structured_input_gemini",
    "4_golden_structured_input_gemini",
    "4_golden_structured_input_gpt",
]

PAIRS_TO_COMPARE = [
    ("4_v2_gemini", "4_structured_input_gemini"),
    ("4_v2_gemini", "4_golden_structured_input_gemini"),
    ("4_structured_input_gemini", "4_golden_structured_input_gemini"),
    ("4_golden_structured_input_gemini", "4_golden_structured_input_gpt"),
]

TOP_N_DIFFS = 20

RESULT_FILES = cr.build_result_files_from_names(RESULT_DATASETS, base_dir=BASE_DIR)
RESULT_FILES

In [ ]:
artifacts = cr.prepare_comparison(
    RESULT_FILES,
    pairs_to_compare=PAIRS_TO_COMPARE,
)

print("Dataset summary")
display(artifacts.dataset_summary)

print("Pairwise overlap summary")
display(artifacts.pairwise_overlap_summary)

for left_name, right_name in PAIRS_TO_COMPARE:
    pair_df = artifacts.comparison_tables[(left_name, right_name)]
    print(f"Detailed comparison pair: {left_name} vs {right_name}")
    print(f"Intersection rows: {len(pair_df)}")
    display(pair_df.head())

In [ ]:
for left_name, right_name in PAIRS_TO_COMPARE:
    pair_df = artifacts.comparison_tables[(left_name, right_name)]
    summary_stats = cr.build_pair_summary_stats(pair_df, left_name, right_name)

    print(f"Summary stats: {left_name} vs {right_name}")
    display(summary_stats)

    if len(pair_df) < 2:
        print("Warning: 当前交集样本数小于 2，相关性统计没有解释意义。")

In [ ]:
for left_name, right_name in PAIRS_TO_COMPARE:
    pair_df = artifacts.comparison_tables[(left_name, right_name)]
    print(f"Plots for: {left_name} vs {right_name}")

    if pair_df.empty:
        print("Skipped plots because the intersection is empty.")
        continue

    plot_artifacts = cr.plot_pair_diagnostics(pair_df, left_name, right_name)

    if plot_artifacts["heatmap"] is None:
        print("Skipped correlation heatmap because the intersection has fewer than 2 rows.")

In [ ]:
for left_name, right_name in PAIRS_TO_COMPARE:
    pair_df = artifacts.comparison_tables[(left_name, right_name)]
    print(f"Top {TOP_N_DIFFS} differences: {left_name} vs {right_name}")

    if pair_df.empty:
        print("Skipped detailed diff tables because the intersection is empty.")
        continue

    top_tables = cr.get_top_differences(pair_df, left_name, right_name, top_n=TOP_N_DIFFS)

    print(f"Top {TOP_N_DIFFS} sentiment_score differences")
    display(top_tables["top_score_diffs"])

    print(f"Top {TOP_N_DIFFS} confidence differences")
    display(top_tables["top_confidence_diffs"])